# Data Transformation in Pyspark

In [0]:
df=spark.read.format('csv').options(header='true').load('/Volumes/pyspark_tutorial/pyspark/data/olist_orders_dataset.csv')

## 1. Schema Handling

In [0]:
from pyspark.sql.functions import * 

### We can define schema by ourself or use inferSchema=True otion

### Schema Defintion Method 1

In [0]:
schema = "order_id string,customer_id string,order_status string, order_purchase_timestamp string, order_approved_at timestamp, order_delivered_carrier_date timestamp, order_delivered_customer_date timestamp, order_estimated_delivery_date timestamp"    


### Schema Definition Method 2

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema2 = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True)
])

In [0]:
df.printSchema()

### We need to change data type of InvoiceDate as it is string

### 1.1 Schema Enformcement By Definition

In [0]:
df=spark.read.format('csv').options(header='true').schema(schema).load('/Volumes/pyspark_tutorial/pyspark/data/olist_orders_dataset.csv')

In [0]:
df=spark.read.format('csv').options(header='true').schema(schema2).load('/Volumes/pyspark_tutorial/pyspark/data/olist_orders_dataset.csv')

### 1.2 Spark Native Schema Enforcement

In [0]:
df = spark.read.format('csv').options(header='true',inferSchema='true').load('/Volumes/pyspark_tutorial/pyspark/data/olist_orders_dataset.csv')

## 2. Data Type Conversion

## 2.1 Cast Method

In [0]:
df.printSchema()

In [0]:
df = df.withColumn("order_purchase_timestamp",col("order_purchase_timestamp").cast("date"))

###2.2 to_timestamp

In [0]:
df=df.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"), "MM-dd-yyyy HH:mm:ss")
)

In [0]:
display(df)

## 3. Column Transformation

In [0]:
df.printSchema()

###3.1 Column Renaming

#### withColumnRenamed vs withColumnsRenamed

The first one can only change single columns but the second one can change dictionary of columns old and new name {x:y,z:c}

In [0]:
df = df.withColumnsRenamed({
    'order_status':'status',
    'order_purchase_timestamp':'purchase',
    'order_approved_at':'approved',
    'order_delivered_carrier_date':'carrier'}
)

In [0]:
df = df.withColumnRenamed(
    'customer_id','cust_id'
)

## 3.2 Add New Columns

In [0]:
df = df.withColumn('Country',lit('Brazil'))

In [0]:
display(df)

In [0]:
display(df.select("status").distinct())

### Read New  Dataset

In [0]:
from pyspark.sql.types import *

In [0]:
schema= StructType([
    StructField('review_id',StringType(),False),
    StructField('order_id',StringType(),False),
    StructField('review_score',IntegerType(),False),
    StructField('review_comment_title',StringType(),False),
    StructField('review_comment_message',StringType(),False),
    StructField('review_creation_date',TimestampType(),False),
    StructField('review_answer_timestamp',TimestampType(),False)
]
)

In [0]:
df2= spark.read.format('csv').options(header='true').schema(schema).load('/Volumes/pyspark_tutorial/pyspark/data/olist_order_reviews_dataset.csv')

In [0]:
display(df2)

### 3.3 Handling Null Values

In [0]:
display(df2.describe())

Note:- If there is not review_comment message, it isn't necessary to keep the rows.
Let us filter rows with null comment

In [0]:
df2.columns

In [0]:
df2= df2.filter(df2.review_comment_message.isNotNull())

In [0]:
display(df2)

### 3.2.1 Count Number of Null Values

In [0]:
df2.filter(df2.review_comment_title.isNull()).count()

In [0]:
df2.where(df2.review_comment_title.isNotNull()).count()

In [0]:
df2.where(df2.review_score.isNotNull()).count()

### 3.2.2 Drop Column 

Note:- Some columns don't add meaning to our analysis so better to drop it

In [0]:
df2= df2.drop('review_comment_title')

In [0]:
df2.columns

### SelectExpr

SelectExpr used to apply complex filter condition in our dropping columns 

In [0]:
display(df2.select(year(col("review_creation_date"))).distinct())

I want the recent year review where our recent year is 2018

In [0]:
display(df2)

In [0]:
from pyspark.sql.functions import *

In [0]:

from pyspark.sql.functions import *

In [0]:
df_view = df2.createOrReplaceTempView("df2")

In [0]:
df2_2=spark.table('df2')

In [0]:
%sql
select 
* from df2

In [0]:
df2_recent = df2_2.selectExpr("review_id", "order_id", "review_score", "review_comment_message", "review_creation_date", "review_answer_timestamp").filter(year(col("review_creation_date")) == 2018)

In [0]:
display(df2_recent)

In [0]:
df2_best_score = df2.selectExpr('review_id','order_id','review_comment_message').filter(col('review_score')==5)

In [0]:
display(df2_best_score)

### 4. Statistics of Our Data

In [0]:
df2.count()

In [0]:
display(df2.describe())

In [0]:
df.select(
    [ 
     sum(col(c).isNull().cast('int')).alias(c)
    for c in df.columns
    ]
).show()

In [0]:
display(df.summary())

### 5. Conditional Transformation

In [0]:
display(df.select(col('status')).distinct())

Note:- we want the whole different status into three. Inprogress, Completed, Quarantined:-cancelled,unavailable

In [0]:
from pyspark.sql.functions import col, lit, when

df = df.withColumn(
    "order_status",
    when(
        col("status").isin(
            "approved", "shipped", "invoiced", "created", "processing"
        ),
        lit("Inprogress")
    )
    .when(
        col("status") == "delivered",
        lit("Complete")
    )
    .otherwise(
        lit("Quarantined")
    )
)

In [0]:
df.show()

**How many complete & inprogress order we have?**

In [0]:
df.groupBy('order_status').count().show()

In [0]:
df.groupby("order_status").count().filter(col('order_status')!='Quarantined').show()

In [0]:
df.columns

#### Multiple filter condition

In [0]:
df.filter(
    (col("order_status") == "Inprogress") & (col("status") =='invoiced') & (year(col("purchase"))== 2017)
).show()

Note:- We want to know the progress of our customer service order which process the request with the aim of understanding the gap between purchase and approved date

#### DateDiff Function

In [0]:
df = df.withColumn(
    'Process_Date',
    date_diff(col("approved"),col("purchase"))
)

In [0]:
df.display()

In [0]:
display(df.select(col("Process_Date")).distinct())

In [0]:
df.select(avg(col("Process_Date")).alias("average_process_date")).show()

### 6. String Transformation

In [0]:
df2 = df2.drop(col('review_comment_title'))

In [0]:
df2.show()

#### Upper

In [0]:
display(df2.withColumn('review_comment_message',upper(col('review_comment_message'))))

#### Lower Function

In [0]:
display(df2.withColumn('review_comment_message',lower(col('review_comment_message'))))

#### InitCap

In [0]:
display(df2.withColumn('review_comment_message',initcap(col('review_comment_message'))))

### 7. Aggeragation Transformation

In [0]:
df.columns

In [0]:
display(df.groupBy("cust_id")
        .agg(count('order_id').alias('Num_of_order'))
        .filter((col('Num_of_order') >= 1)))
   
   

In [0]:
display(df.groupBy("order_status")
  .agg(count('*').alias('Num_of_order')))


#### Read New Dataset

In [0]:
schema = StructType(
    [
        StructField("order_id",StringType(),False),
        StructField("payment_sequential",IntegerType(),True),
        StructField("payment_type",StringType(),True),
        StructField("payment_installments",IntegerType(),True),
        StructField("payment_value",FloatType(),False)

    ]
)

In [0]:
df3= spark.read.format('csv').options(header='true').schema(schema).load('/Volumes/pyspark_tutorial/pyspark/data/olist_order_payments_dataset.csv')

In [0]:
df3.where(df3.payment_installments.isNull()).count()

In [0]:
display(
    df3.groupBy('order_id')
        .agg(count('*').alias('order_count'))
        .filter(col('order_count') > 1)
        )
        

In [0]:
display(df3.where(df3.order_id =='3c58bffb70dcf45f12bdf66a3c215905').orderBy('payment_sequential'))

In [0]:
display(
    df3.groupBy('order_id')
    .agg(floor(sum(col('payment_value')),2).alias('Total_Payment'))
    .orderBy(col('Total_Payment').desc())
)